# CS-781 Phase 7 (Prior Case Retrieval) - `p7_gqa2`

**Phase-1 model:** dense GQA + RoPE decoder (111.1M params).  **This notebook:** Round 2: the GQA with the best Round-1 design choices combined (uniform chunk selection, ListNet, LoRMA, 6 epochs), without projection, and with 8 training chunks.

The Phase-1 model becomes a **case encoder** (bi-encoder): a case is read in chunks of 512 tokens (training: 4 chunks, first 2 + last 2; indexing: 8), each chunk is mean-pooled, the chunk vectors are averaged, projected to 256 dimensions and normalised; relevance = cosine similarity, and the 7 most similar cases of the split's pool are returned (the query itself excluded). Training: InfoNCE over the batch (8 queries; for each a relevant case + 1 hard negative from the TF-IDF ranking; the other queries' cases act as in-batch negatives; other relevant cases are never used as negatives). After every epoch the 118 dev queries are ranked against the dev pool (Micro-F1@7, as `evaluate_phase7.py`), the best epoch is kept. Epoch 0 = the Phase-1 weights before any training. At the end: dev and test indexed, a hybrid with TF-IDF (dense + lambda x TF-IDF, lambda chosen on dev) is also scored, and submission zips (`prediction.json`) are written for both. Every finished setup is cached in the bucket (Run All again resumes).

**Kaggle:** GPU T4, Internet on, secrets `HF` and `HF_WRITE`. `QUICK_TEST = True` in cell 1 = a few-minute check. Hyper-parameters are placeholders proposed by the assistant.

## 1. Settings

In [1]:
# [P7-PRELUDE:settings]
import json
# =====================================================================================================
# SETTINGS -- every value is a placeholder you can change. Hyper-parameter values were proposed by the assistant and are NOT tuned.
# =====================================================================================================
SWEEP_NAME = "p7_gqa2"
MODEL_KEY = "gqa160"
PHASE1_CHECKPOINT = "phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt"
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"
PEFT_TARGETS = ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj"]
IA3_OUT = ["attn.k_proj", "attn.v_proj"]
IA3_IN = ["ff.2"]

ATTENTION = "causal"   # only used by the Phase-1 decoder code below
MAX_DOC_TOKENS = 16384  # a longer case keeps its first and last half of this many tokens (94% of the cases are shorter)

BASE = dict(
    # text: "keep" = the case as given; "strip" = without the citation strings of the Supreme-Court header (CITATION / CITATOR INFO) and reporter citations in the body
    HEADER="keep",
    # long cases: chunks of WINDOW tokens (STRIDE < WINDOW = overlapping), TRAIN_CHUNKS of them per case while training, INFER_CHUNKS when indexing / ranking; SELECT chooses which
    WINDOW=512, STRIDE=512, TRAIN_CHUNKS=4, INFER_CHUNKS=8, SELECT="head_tail", HEAD_CHUNKS=2,
    # representation: token pooling inside a chunk (mean | last | mean+last | max), chunk pooling (mean | max | attn | first | maxsim), projection size (0 = none), similarity (cos | dot)
    POOL="mean", DOC_POOL="mean", PROJ_DIM=256, SIM="cos", TAU=0.05,
    # objective: infonce (in-batch + own negatives) | infonce_own (own negatives only) | listwise (ListNet over all positives) | pairwise (softplus) | margin (hinge)
    OBJECTIVE="infonce", MARGIN=0.2, LIST_POS=3,
    # negatives per query: NEG = hard (from the TF-IDF top HARD_TOP of the query, not relevant) | random | mixed
    NEG="hard", N_NEG=1, HARD_TOP=50,
    TECHNIQUE="lorma", RANK=16, ALPHA=32, LORA_DROPOUT=0.0, PEFT_TARGETS=PEFT_TARGETS, LORMA_VARIANT="plus",
    ADALORA_INIT_RANK=24, ADALORA_TARGET_RANK=8, ADALORA_DELTA_T=10, ADALORA_ORTH=0.1,
    IA3_OUT=IA3_OUT, IA3_IN=IA3_IN, ADAPTER_KIND="houlsby", ADAPTER_R=64, PARTIAL_LAYERS=2, PROMPT_LEN=20, PREFIX_LEN=16,
    # training recipe: an epoch = every training query once, BATCH queries per optimizer step
    BATCH=8, ROW_GROUP=8, CKPT=True, INFER_BATCH=8, EPOCHS=6, LR=3e-4, HEAD_LR=1e-3, WEIGHT_DECAY=0.01, WARMUP_FRAC=0.1, GRAD_CLIP=1.0, AMP="fp16", SEED=0,
    TIME_BUDGET_MIN=None, EVAL_EPOCH0=True, RANDOM_INIT=False,
    # knowledge distillation from a teacher retriever's case vectors (file p7emb_*.pt written by a SAVE_TRAIN_EMB setup): scores = KL over the batch's
    # similarity distribution, embed = match the teacher's case vectors, both
    KD_TEACHER=None, KD_MODE="scores", KD_ALPHA=0.5, KD_TEMPERATURE=1.0, TEACHER_TAU=0.05,
    SAVE_TRAIN_EMB=False, SAVE_WEIGHTS=False)     # SAVE_WEIGHTS = upload the trained parameters (p7w_*.pt); off by default to save bucket space (scores and case vectors are always saved)
LR_BY_TECH = dict(full_ft=2e-5, partial=5e-5, ia3=3e-3, prompt=3e-3, prefix=3e-3, bitfit=1e-3, adapters=5e-4, frozen=3e-4)
HYBRID_LAMBDAS = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]      # dense + lambda * TF-IDF (z-scored per query); lambda chosen on dev

SWEEP = [   # (name, settings that differ from BASE). Order = run order.
    ("gqa_uni_list", dict(TECHNIQUE='lorma', SELECT='uniform', OBJECTIVE='listwise')),
    ("gqa_uni_list_proj0", dict(TECHNIQUE='lorma', SELECT='uniform', OBJECTIVE='listwise', PROJ_DIM=0)),
    ("gqa_uni_list_8ch", dict(TECHNIQUE='lorma', SELECT='uniform', OBJECTIVE='listwise', TRAIN_CHUNKS=8, BATCH=4)),
]
TEACHER_WAIT_MIN = 0         # a KD setup waits up to this long for the teacher's file to appear in the bucket
SUBMIT = True                 # write submission zips (dense, and hybrid with TF-IDF) for every finished setup
MAX_SWEEP_HOURS = 10.5            # do not START a setup that would not finish within this many hours (finished setups are cached: Run All again resumes)
QUICK_TEST = False                # True = tiny end-to-end check (few queries, 1 epoch, short chunks, first 2 setups)
if QUICK_TEST:
    SWEEP = [(n, dict(o, EPOCHS=1)) for n, o in SWEEP[:2]]
    SWEEP = [(n, dict(o, KD_TEACHER=o["KD_TEACHER"].replace("__", "_quicktest__", 1)) if o.get("KD_TEACHER") else o) for n, o in SWEEP]
    BASE.update(WINDOW=64, STRIDE=64, TRAIN_CHUNKS=2, INFER_CHUNKS=3, BATCH=4, PROJ_DIM=32)
    TEACHER_WAIT_MIN = 0
    SWEEP_NAME += "_quicktest"
print(f"{SWEEP_NAME}: {len(SWEEP)} setups:", [n for n, _ in SWEEP])

p7_gqa2: 3 setups: ['gqa_uni_list', 'gqa_uni_list_proj0', 'gqa_uni_list_8ch']


## 2. Set-up: login, bucket

In [2]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.4.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.4.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


## 3. Phase-1 BPE tokenizer

In [3]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


## 4. Phase-1 model code (unchanged)

In [4]:
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"   # as in the Phase-1 run
CONTEXT_LEN = 2048

class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# ---- Settings specific to this notebook ------------------------------
GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.fc_out(h)

## 5. Backbone wrapper (unchanged from Phase 2)

In [5]:
# ===================== Backbone wrapper for the Phase-1 decoder models (GQA / MoE CachedDecoderModel) =====================
# The Phase-1 model code above is UNCHANGED. For NER we only (a) run its blocks and take the hidden states instead of the
# next-token logits, and (b) optionally replace the attention's forward with one that can drop the causal mask
# ("bidirectional") and/or prepend learned prefix keys/values (prefix tuning). With both options off it calls the
# original forward unchanged.

_ORIG_GQA_FORWARD = CachedGQAttention.forward


def ner_gqa_forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
    bidir = getattr(self, "ner_bidirectional", False)
    pk = getattr(self, "prefix_k", None)
    if not bidir and pk is None:
        return _ORIG_GQA_FORWARD(self, x, key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    if rope is not None:
        q = apply_rope(q, *rope)
        k = apply_rope(k, *rope)
    P = 0
    if pk is not None:                                          # learned prefix keys/values (no position: never rotated)
        P = pk.shape[1]
        k = torch.cat([pk.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
        v = torch.cat([self.prefix_v.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
    if self.group_size > 1:
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
    if bidir and P == 0 and key_padding_mask is None:          # full-attention fast path (no mask tensor at all)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
        return self.out_proj(out)
    pos = torch.arange(T, device=x.device)
    allowed = torch.ones(T, T, dtype=torch.bool, device=x.device) if bidir else (pos.unsqueeze(1) >= pos.unsqueeze(0))
    allowed = allowed.unsqueeze(0).expand(B, T, T)
    if key_padding_mask is not None:
        allowed = allowed & key_padding_mask.unsqueeze(1)
        allowed = allowed | torch.eye(T, dtype=torch.bool, device=x.device).unsqueeze(0)
    if P:
        allowed = torch.cat([torch.ones(B, T, P, dtype=torch.bool, device=x.device), allowed], dim=2)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed.unsqueeze(1))
    out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
    return self.out_proj(out)


CachedGQAttention.forward = ner_gqa_forward


class DecoderBackbone(nn.Module):
    def __init__(self, lm, bidirectional=False):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        self.causal = not bidirectional
        for blk in lm.blocks:
            blk.attn.ner_bidirectional = bidirectional
        if isinstance(lm.fc_out, nn.Sequential):                # the final LayerNorm is kept; the vocab projection is dropped
            self.final_norm = lm.fc_out[0]
        else:
            self.final_norm = nn.Identity()
        lm.fc_out = nn.Identity()
        self.d_model = lm.token_emb.embedding_dim
        self.max_len = lm.rope_cos.shape[0] if lm.use_rope else lm.pos_emb.num_embeddings

    def layer_list(self):
        return list(self.lm.blocks)

    def final_modules_params(self):
        return list(self.final_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        d = self.d_model
        if kind == "houlsby":                                   # after attention AND after the feed-forward, every block
            return [(m, d) for blk in self.lm.blocks for m in (blk.attn, blk.ff)]
        if kind == "pfeiffer":                                  # after the feed-forward only
            return [(blk.ff, d) for blk in self.lm.blocks]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        for blk in self.lm.blocks:
            a = blk.attn
            a.prefix_k = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)
            a.prefix_v = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)

    def features(self, ids, mask, soft_prompt=None):
        lm = self.lm
        h = lm.token_emb(ids)
        B = ids.shape[0]
        P = 0
        if soft_prompt is not None:
            P = soft_prompt.shape[0]
            h = torch.cat([soft_prompt.unsqueeze(0).expand(B, -1, -1).to(h.dtype), h], dim=1)
            mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=mask.device), mask], dim=1)
        T = h.shape[1]
        assert T <= self.max_len, f"window ({T} incl. {P} prompt tokens) longer than the model's {self.max_len} positions"
        pos = torch.arange(T, device=ids.device)
        rope = None
        if lm.use_rope:
            rope = (lm.rope_cos[pos].unsqueeze(0).unsqueeze(0), lm.rope_sin[pos].unsqueeze(0).unsqueeze(0))
        else:
            h = h + lm.pos_emb(pos).unsqueeze(0)
        # right padding: with causal attention real tokens never see the pads, so no mask is needed (fast fused kernel);
        # bidirectional attention needs the padding mask
        kpm = None if (self.causal or bool(mask.all())) else mask
        for blk in lm.blocks:
            h = blk(h, key_padding_mask=kpm, rope=rope)
        h = self.final_norm(h)
        return h[:, P:]


@torch.no_grad()
def lm_sanity_loss(lm, docs, device, n_windows=6, length=1024):
    """Next-token loss of the loaded Phase-1 model on a few NER-document windows (BEFORE the LM head is dropped).
    A correctly loaded model gives roughly its Phase-1 held-out loss (~2.5-3.5 nats); random weights give ~10.6."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:length + 1], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < 2:
            continue
        logits = lm(ids[:, :-1]).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="sum").item()
        cnt += ids.shape[1] - 1
    return tot / max(cnt, 1)


def build_phase1_lm():
    """The Phase-1 dense GQA model exactly as trained (111.1M params, no final LayerNorm), with its trained weights."""
    torch.manual_seed(0)
    lm = CachedDecoderModel(VOCAB, d_model=768, n_heads=12, n_layers=8, d_ff=3072, context_len=CONTEXT_LEN)
    lm.load_state_dict(PHASE1_STATE, strict=True)
    return lm


def make_backbone(lm):
    return DecoderBackbone(lm, bidirectional=(ATTENTION == "bidirectional"))

## 6. Phase-7 engine: case text, chunked case encoder, objectives, negatives, KD, Micro-F1@7, hybrid, submission

In [6]:
# ======================================================================================================================
# Phase 7 (Prior Case Retrieval) engine: case text (with / without the citation header), chunked case encoder built on a
# Phase-1 backbone, query-candidate similarity, contrastive / pairwise / listwise objectives with in-batch and hard
# negatives, knowledge distillation (teacher similarity scores or embeddings), Micro-F1@7 evaluation, TF-IDF hybrid,
# submission files. Everything model-specific comes from the Phase-1 / Phase-2 cells above (backbone.features()).
# ======================================================================================================================
import math, time, random, json, os, re, zipfile, collections, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
K_EVAL = 7


# ----------------------------------------------------------------------------------------------------------------------
# 1. Case text; the Supreme-Court header (CITATION / CITATOR INFO) leaks citation links -> HEADER="strip" removes them
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(v):
    if isinstance(v, dict):
        for k in ("text", "sentences", "document", "content"):
            if k in v: return doc_text(v[k])
        return ""
    if isinstance(v, list):
        return " ".join(x if isinstance(x, str) else doc_text(x) for x in v)
    return str(v)


_HDR_BLOCK = re.compile(r"(CITATION\s*:.*?|CITATOR\s+INFO\s*:.*?)(?=\bACT\s*:|\bHEADNOTE\s*:|\bJUDGMENT\b|\bCITATOR\s+INFO\b|$)", re.S)
_REPORTER = re.compile(r"\(?\b(?:19|20)\d{2}\)?\s*(?:\(\s*\d+\s*\)\s*)?(?:AIR|SCR|SCC|SCALE|JT|SC|Cri\s*LJ|CriLJ|ITR|SCJ|MLJ|Bom\s*LR|ALL|Cal|Mad|Bom|Ker|Pat|Del)\b"
                       r"\.?\s*(?:\(\s*\d+\s*\)\s*)?(?:Supl\.?|Supp\.?|SC)?\s*\(?\d*\)?\s*\d{1,5}(?:\s*\(\s*[\d,\s]+\))?")
_REPORTER2 = re.compile(r"\bAIR\s+(?:19|20)\d{2}\s+[A-Za-z]{2,6}\s+\d{1,5}")
_REPORTER4 = re.compile(r"\b(?:19|20)\d{2}\s+Supp?l?\.?\s*(?:\(\s*\d+\s*\)\s*)?(?:SCR|SCC)\s*\d{1,5}")
_REPORTER3 = re.compile(r"[\(\[]\s*(?:19|20)\d{2}\s*[\)\]]\s*\d{0,3}\s*(?:SCC|SCR|SCALE|JT|SC|ITR|Cri\s*LJ|CriLJ|SCJ|MLJ)\b\.?\s*(?:\(\s*\w+\s*\)\s*)?\d{1,5}")


def strip_header(t):
    """Remove the citation strings that link cases: the CITATION / CITATOR INFO blocks of the Supreme-Court header (only
    in the first 4,000 characters) and every reporter citation in the body (e.g. '1964 AIR 1545', 'AIR 1954 SC 210',
    '(1990) 1 SCC 445')."""
    head, body = t[:4000], t[4000:]
    head = _HDR_BLOCK.sub(" ", head)
    t = head + body
    t = _REPORTER2.sub(" <CITATION> ", t)
    t = _REPORTER3.sub(" <CITATION> ", t)
    t = _REPORTER4.sub(" <CITATION> ", t)
    t = _REPORTER.sub(" <CITATION> ", t)
    return re.sub(r"\s+", " ", t)


def case_text(v, mode):
    t = doc_text(v)
    return strip_header(t) if mode == "strip" else t


_OWN = [re.compile(r"\b((?:19|20)\d{2})\s+AIR\s+(\d{1,5})"), re.compile(r"\bAIR\s+((?:19|20)\d{2})\s+SC\s*(\d{1,5})")]
_CITATOR_REF = re.compile(r"\b((?:19|20)\d{2})\s*SC\s*(\d{1,5})")


def citation_keys(t):
    """(own AIR citations, AIR citations listed in its CITATOR INFO = later judgments that cite it) from the header."""
    h = t[:5000]; own, refs = set(), set()
    m = re.search(r"CITATION\s*:(.*?)(?:CITATOR\s+INFO|ACT\s*:|HEADNOTE|$)", h, re.S)
    if m:
        for p in _OWN: own |= {(a, str(int(b))) for a, b in p.findall(m.group(1))}
    m = re.search(r"CITATOR\s+INFO\s*:(.*?)(?:ACT\s*:|HEADNOTE|JUDGMENT|$)", h, re.S)
    if m:
        refs = {(a, str(int(b))) for a, b in _CITATOR_REF.findall(m.group(1))}
    return own, refs


def encode_case(tok, text, max_tokens):
    ids = tok.encode(text)
    if len(ids) > max_tokens:
        h = max_tokens // 2; ids = ids[:h] + ids[-(max_tokens - h):]
    return np.asarray(ids, dtype=np.uint16)


# ----------------------------------------------------------------------------------------------------------------------
# 2. Chunks and batches
# ----------------------------------------------------------------------------------------------------------------------
def make_spans(n, W, S, m, how, head_chunks):
    if n <= W:
        return [(0, max(n, 1))]
    starts = list(range(0, n - W + 1, max(1, S)))
    if starts[-1] + W < n: starts.append(n - W)
    spans = [(s, s + W) for s in starts]; K = len(spans)
    if m is None or K <= m or how == "all":
        return spans
    if how == "first": idx = list(range(m))
    elif how == "last": idx = list(range(K - m, K))
    elif how == "head_tail":
        h = min(head_chunks, m - 1) if m > 1 else 0; idx = list(range(h)) + list(range(K - (m - h), K))
    elif how == "uniform": idx = sorted(set(int(round(x)) for x in np.linspace(0, K - 1, m)))
    else: raise ValueError(how)
    return [spans[i] for i in idx]


def collate_docs(token_lists, C, n_chunks, pad_id=0):
    rows, counts = [], []
    for ids in token_lists:
        sp = make_spans(len(ids), C["WINDOW"], C["STRIDE"], n_chunks, C["SELECT"], C["HEAD_CHUNKS"])
        for s, e in sp:
            r = ids[s:e]; rows.append(r if len(r) else np.zeros(1, dtype=np.uint16))
        counts.append(len(sp))
    R, T = len(rows), max(len(r) for r in rows)
    X = torch.full((R, T), pad_id, dtype=torch.long); M = torch.zeros((R, T), dtype=torch.bool)
    for r, a in enumerate(rows):
        X[r, :len(a)] = torch.from_numpy(a.astype(np.int64)); M[r, :len(a)] = True
    K = max(counts)
    cm = torch.arange(K).unsqueeze(0) < torch.tensor(counts).unsqueeze(1)
    return dict(ids=X, mask=M, chunk_mask=cm)


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 3. Case encoder: Phase-1 backbone -> chunk vectors (token pooling) -> case vector (chunk pooling) -> projection
# ----------------------------------------------------------------------------------------------------------------------
class CaseEncoder(nn.Module):
    """Attribute names shared with the Phase-3/5 model (backbone, soft_prompt, head_parameters) so the technique code works unchanged.
    POOL (tokens of a chunk): mean | last | mean+last | max.  DOC_POOL (chunks of a case): mean | max | attn | first | maxsim
    (maxsim = late interaction: the case is kept as its chunk vectors; score = mean over query chunks of the best candidate chunk)."""

    def __init__(self, backbone, d_model, C):
        super().__init__()
        self.backbone, self.soft_prompt = backbone, None
        self.pool, self.doc_pool, self.sim = C["POOL"], C["DOC_POOL"], C["SIM"]
        self.row_group, self.ckpt = C["ROW_GROUP"], C.get("CKPT", True)
        if self.ckpt and any(getattr(m, "bias_rate", 0) > 0 or (hasattr(m, "tau") and getattr(type(m)._route, "__name__", "") != "_mor_route_frozen") for m in backbone.modules()):
            self.ckpt = False
        d = d_model * (2 if self.pool == "mean+last" else 1)
        self.attn = nn.Linear(d, 1) if self.doc_pool == "attn" else None
        self.proj = nn.Linear(d, C["PROJ_DIM"]) if C["PROJ_DIM"] else nn.Identity()
        if C["PROJ_DIM"] and C.get("PROJ_INIT", "random") == "random":
            nn.init.normal_(self.proj.weight, std=1.0 / math.sqrt(d)); nn.init.zeros_(self.proj.bias)
        self.out_dim = C["PROJ_DIM"] or d

    def head_parameters(self):
        return list(self.proj.parameters()) + (list(self.attn.parameters()) if self.attn is not None else [])

    def _pool_rows(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt).float()
        m = mask.unsqueeze(-1).float(); parts = []
        if self.pool in ("mean", "mean+last"):
            parts.append((h * m).sum(1) / m.sum(1).clamp(min=1))
        if self.pool in ("last", "mean+last"):
            parts.append(h[torch.arange(h.shape[0], device=h.device), mask.sum(1) - 1])
        if self.pool == "max":
            parts.append(h.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values)
        return torch.cat(parts, -1)

    def chunk_vectors(self, b):
        ids, mask = b["ids"], b["mask"]
        trainable = self.training and (any(p.requires_grad for p in self.backbone.parameters()) or self.soft_prompt is not None)
        out = []
        for a in range(0, ids.shape[0], self.row_group):
            i, m = ids[a:a + self.row_group], mask[a:a + self.row_group]
            L = int(m.sum(1).max()); i, m = i[:, :L], m[:, :L]
            if not trainable:
                with torch.no_grad(): out.append(self._pool_rows(i, m))
            elif self.ckpt: out.append(checkpoint(self._pool_rows, i, m, use_reentrant=False))
            else: out.append(self._pool_rows(i, m))
        return torch.cat(out, 0)

    def forward(self, b):
        """-> (B, D) case vectors, or (B, K, D) chunk vectors + (B, K) mask for maxsim."""
        v = self.chunk_vectors(b); cm = b["chunk_mask"]
        vp = v.new_zeros(cm.shape[0], cm.shape[1], v.shape[1]); vp[cm] = v
        if self.doc_pool == "maxsim":
            z = self.proj(vp)
            return (F.normalize(z, dim=-1) if self.sim == "cos" else z), cm
        m = cm.unsqueeze(-1).float()
        if self.doc_pool == "mean": d = (vp * m).sum(1) / m.sum(1).clamp(min=1)
        elif self.doc_pool == "max": d = vp.masked_fill(~cm.unsqueeze(-1), -1e4).max(1).values
        elif self.doc_pool == "first": d = vp[:, 0]
        elif self.doc_pool == "attn":
            a = self.attn(vp).squeeze(-1).masked_fill(~cm, -1e4); d = (torch.softmax(a, -1).unsqueeze(-1) * vp).sum(1)
        else: raise ValueError(self.doc_pool)
        z = self.proj(d)
        return F.normalize(z, dim=-1) if self.sim == "cos" else z


def sim_matrix(Q, Cn, kind):
    """Q, Cn: case vectors (B, D) / (N, D), or (vecs, mask) pairs for maxsim."""
    if isinstance(Q, tuple):
        (q, qm), (c, cmask) = Q, Cn
        s = torch.einsum("aid,bjd->abij", q, c)                                   # (B, N, Kq, Kc)
        s = s.masked_fill(~cmask[None, :, None, :], -1e4).max(-1).values          # best candidate chunk
        qmf = qm.float()[:, None, :]
        return (s * qmf).sum(-1) / qmf.sum(-1).clamp(min=1)
    s = Q @ Cn.T
    return s if kind == "cos" else s / math.sqrt(Q.shape[1])


# ----------------------------------------------------------------------------------------------------------------------
# 4. Training examples, objectives, KD
# ----------------------------------------------------------------------------------------------------------------------
def sample_batch(qids, rel, pool, hard, C, rng):
    """For each query: LIST_POS positives (InfoNCE / pairwise use the first) and N_NEG negatives (random / hard / mixed)."""
    pos_lists, neg_lists = [], []
    for q in qids:
        r = [c for c in rel[q] if c != q and c in pool]
        pos_lists.append(rng.sample(r, min(len(r), C["LIST_POS"] if C["OBJECTIVE"] == "listwise" else 1)))
        bad = set(rel[q]) | {q}; negs = []
        for k in range(C["N_NEG"]):
            how = C["NEG"] if C["NEG"] != "mixed" else ("hard" if k % 2 == 0 else "random")
            if how == "hard" and hard.get(q):
                cand = [c for c in hard[q][:C["HARD_TOP"]] if c not in bad and c not in negs]
            else:
                cand = None
            if cand:
                negs.append(rng.choice(cand))
            else:
                for _ in range(50):
                    c = rng.choice(pool)
                    if c not in bad and c not in negs: negs.append(c); break
        neg_lists.append(negs)
    return pos_lists, neg_lists


def retrieval_loss(S, B, cand_ids, qids, pos_lists, neg_lists, rel, C):
    """S: (B, N) query x candidate scores for this batch (candidates = every positive and negative of every query, in order)."""
    tau = C["TAU"]; dev = S.device
    relset = [set(rel[q]) | {q} for q in qids]
    pos_mask = torch.tensor([[c in relset[i] and c != qids[i] for c in cand_ids] for i in range(B)], device=dev)
    own = [set(p) | set(n) for p, n in zip(pos_lists, neg_lists)]
    if C["OBJECTIVE"] in ("infonce", "infonce_own"):
        tgt = torch.tensor([cand_ids.index(p[0]) for p in pos_lists], device=dev)
        keep = torch.ones_like(S, dtype=torch.bool)
        keep &= ~(pos_mask & (torch.arange(S.shape[1], device=dev)[None] != tgt[:, None]))       # other positives are not negatives
        if C["OBJECTIVE"] == "infonce_own":                                                        # no in-batch negatives
            keep &= torch.tensor([[c in own[i] for c in cand_ids] for i in range(B)], device=dev)
        return F.cross_entropy((S / tau).masked_fill(~keep, -1e4), tgt)
    if C["OBJECTIVE"] == "listwise":                                                               # ListNet: target = uniform over all positives in the batch list
        t = pos_mask.float(); t = t / t.sum(1, keepdim=True).clamp(min=1)
        return -(t * F.log_softmax(S / tau, -1)).sum(1).mean()
    if C["OBJECTIVE"] in ("pairwise", "margin"):                                                   # own positive vs each own negative
        terms = []
        for i in range(B):
            p = cand_ids.index(pos_lists[i][0])
            for n in neg_lists[i]:
                d = S[i, cand_ids.index(n)] - S[i, p]
                terms.append(F.softplus(d / tau) if C["OBJECTIVE"] == "pairwise" else F.relu(C["MARGIN"] + d))
        return torch.stack(terms).mean()
    raise ValueError(C["OBJECTIVE"])


def kd_terms(S, qv, cv, qids, cand_ids, teacher, C):
    """KD_MODE scores: KL(teacher || student) over the batch's candidate list (teacher similarity distribution);
    embed: cosine distance between student and teacher case vectors (intermediate representation, PROJ_DIM must match)."""
    out = 0.0
    TQ = torch.stack([teacher[q] for q in qids]).to(S.device).float(); TC = torch.stack([teacher[c] for c in cand_ids]).to(S.device).float()
    if C["KD_MODE"] in ("scores", "both"):
        St = TQ @ TC.T; T = C["KD_TEMPERATURE"]
        out = out + F.kl_div(F.log_softmax(S / C["TAU"] / T, -1), F.softmax(St / C["TEACHER_TAU"] / T, -1), reduction="batchmean") * T * T
    if C["KD_MODE"] in ("embed", "both"):
        assert qv.shape[-1] == TQ.shape[-1], "embedding KD needs PROJ_DIM equal to the teacher's"
        out = out + (1 - F.cosine_similarity(qv, TQ, -1)).mean() + (1 - F.cosine_similarity(cv, TC, -1)).mean()
    return out


# ----------------------------------------------------------------------------------------------------------------------
# 5. Encoding, ranking, scores
# ----------------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(model, toks, ids, C, device, n_chunks, bs=None):
    """-> {id: vector (D,) or (chunk vectors, mask)} on CPU, fp16."""
    model.eval(); out = {}; bs = bs or C["INFER_BATCH"]
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    order = sorted(ids, key=lambda i: len(toks[i]))
    for a in range(0, len(order), bs):
        grp = order[a:a + bs]
        b = to_device(collate_docs([toks[i] for i in grp], C, n_chunks), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            v = model(b)
        if isinstance(v, tuple):
            for k, i in enumerate(grp): out[i] = (v[0][k].half().cpu(), v[1][k].cpu())
        else:
            for k, i in enumerate(grp): out[i] = v[k].float().half().cpu()
    return out


def score_matrix(E, qids, cids, kind, device, chunk=256):
    if isinstance(E[qids[0]], tuple):
        Kq = max(E[q][0].shape[0] for q in qids); Kc = max(E[c][0].shape[0] for c in cids)
        def stack(ids, K):
            v = torch.zeros(len(ids), K, E[ids[0]][0].shape[-1]); m = torch.zeros(len(ids), K, dtype=torch.bool)
            for i, x in enumerate(ids): k = E[x][0].shape[0]; v[i, :k] = E[x][0].float(); m[i, :k] = E[x][1][:k]
            return v.to(device), m.to(device)
        Cv = stack(cids, Kc); rows = []
        for a in range(0, len(qids), 16):
            rows.append(sim_matrix(stack(qids[a:a + 16], Kq), Cv, kind).cpu())
        return torch.cat(rows).numpy()
    Q = torch.stack([E[q].float() for q in qids]).to(device); Cm = torch.stack([E[c].float() for c in cids]).to(device)
    return torch.cat([sim_matrix(Q[a:a + chunk], Cm, kind).cpu() for a in range(0, len(qids), chunk)]).numpy()


def rank_from_scores(S, qids, cids, k=200):
    pos = {c: j for j, c in enumerate(cids)}; S = S.copy()
    for i, q in enumerate(qids):
        if q in pos: S[i, pos[q]] = -1e9                                       # a case is never its own prior case
    top = np.argsort(-S, 1)[:, :k]
    return {q: [cids[j] for j in top[i]] for i, q in enumerate(qids)}


def prf_at(gold, ranked, k=K_EVAL):
    tp = fp = fn = 0
    for q, g in gold.items():
        g = set(g); p = set(ranked[q][:k]); tp += len(g & p); fp += len(p - g); fn += len(g - p)
    return dict(f1=2 * tp / max(1, 2 * tp + fp + fn), p=tp / max(1, tp + fp), r=tp / max(1, tp + fn), tp=tp, fp=fp, fn=fn)


def retrieval_scores(gold, ranked):
    m = prf_at(gold, ranked)
    tot = sum(len(g) for g in gold.values())
    for k in (20, 50, 100, 200):
        m[f"r@{k}"] = sum(len(set(g) & set(ranked[q][:k])) for q, g in gold.items()) / max(1, tot)
    rr = []
    for q, g in gold.items():
        g = set(g); r = next((i + 1 for i, c in enumerate(ranked[q]) if c in g), None); rr.append(1 / r if r else 0.0)
    m["mrr"] = float(np.mean(rr)) if rr else 0.0
    return m


def zscore_rows(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)


def hybrid_scores(Sd, St, lam):
    return zscore_rows(Sd) + lam * zscore_rows(St)


# ----------------------------------------------------------------------------------------------------------------------
# 6. Training loop
# ----------------------------------------------------------------------------------------------------------------------
def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup: return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith((".tau", ".tau_ready", ".ada_mask")): st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


def train_retriever(model, data, C, device, evaluate, log=print, teacher=None):
    """data: dict(qids (training queries with >=1 relevant), rel, pool, toks, hard). evaluate(model) -> dict with 'f1'.
    An epoch = every training query once (shuffled), BATCH queries per step."""
    rng = random.Random(C["SEED"]); torch.manual_seed(C["SEED"])
    params_bb = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("proj.", "attn."))]
    params_hd = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith(("proj.", "attn."))]
    groups = [g for g in (dict(params=params_bb, lr=C["LR"]), dict(params=params_hd, lr=C["HEAD_LR"])) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    qids = list(data["qids"]); steps_ep = math.ceil(len(qids) / C["BATCH"]); total = steps_ep * C["EPOCHS"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    hist, best, best_state, best_ep = [], -1.0, None, 0
    allp = [p for g in groups for p in g["params"]]
    t0 = time.time(); n_docs = 0
    log(f"training: {len(qids)} queries per epoch, {C['BATCH']} per step ({steps_ep} steps), {C['EPOCHS']} epochs; objective {C['OBJECTIVE']}, {C['N_NEG']} {C['NEG']} negative(s) per query"
        + (" + in-batch" if C["OBJECTIVE"] in ("infonce", "listwise") else "") + f"; {C['TRAIN_CHUNKS']} chunks per case in training, {C['INFER_CHUNKS']} at inference; "
        + f"{sum(p.numel() for p in allp):,} trainable parameters" + (f"; KD ({C['KD_MODE']}, alpha {C['KD_ALPHA']})" if teacher is not None else ""))
    r0 = None
    if C.get("EVAL_EPOCH0"):      # the Phase-1 representation as it is (+ the untrained projection): how well does it transfer without training?
        r0 = evaluate(model); log(f"  epoch 0 (before training, Phase-1 weights as they are): dev F1@7 {r0['f1']:.4f} R@100 {r0['r@100']:.3f} MRR {r0['mrr']:.4f}")
    for ep in range(1, C["EPOCHS"] + 1):
        model.train(); rng.shuffle(qids); run, nb, t_ep = 0.0, 0, time.time()
        for a in range(0, len(qids), C["BATCH"]):
            bq = qids[a:a + C["BATCH"]]
            pos_l, neg_l = sample_batch(bq, data["rel"], data["pool"], data["hard"], C, rng)
            cand = list(dict.fromkeys([c for p, n in zip(pos_l, neg_l) for c in p + n]))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                bqv = to_device(collate_docs([data["toks"][q] for q in bq], C, C["TRAIN_CHUNKS"]), device)
                bcv = to_device(collate_docs([data["toks"][c] for c in cand], C, C["TRAIN_CHUNKS"]), device)
                qv, cv = model(bqv), model(bcv)
                S = sim_matrix(qv, cv, C["SIM"]).float()
                loss = retrieval_loss(S, len(bq), cand, bq, pos_l, neg_l, data["rel"], C)
                if teacher is not None and C["KD_ALPHA"] > 0:
                    loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_terms(S, qv if not isinstance(qv, tuple) else qv[0].mean(1),
                                                                                cv if not isinstance(cv, tuple) else cv[0].mean(1), bq, cand, teacher, C)
                if hasattr(model, "extra_loss"): loss = loss + model.extra_loss()
            n_docs += len(bq) + len(cand)
            if not torch.isfinite(loss): continue
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            if hasattr(model, "on_before_step"): model.on_before_step(len(hist) * steps_ep + nb, total)
            torch.nn.utils.clip_grad_norm_(allp, C["GRAD_CLIP"]); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); nb += 1
        t_tr = time.time() - t_ep; t_e = time.time(); r = evaluate(model)
        row = dict(epoch=ep, train_loss=round(run / max(1, nb), 4), dev_f1=round(r["f1"], 4), dev_p=round(r["p"], 4), dev_r=round(r["r"], 4),
                   dev_r100=round(r["r@100"], 3), dev_mrr=round(r["mrr"], 4), train_min=round(t_tr / 60, 1), eval_min=round((time.time() - t_e) / 60, 1))
        hist.append(row); log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if r["f1"] > best:
            best, best_ep, best_state = r["f1"], ep, trainable_state(model)
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget reached after epoch {ep}"); break
    train_secs = sum(h["train_min"] for h in hist) * 60
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(epoch0=r0, history=hist, best_epoch=best_ep, best_dev_f1=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, docs_per_sec=round(n_docs / max(train_secs, 1e-9), 2))


# ----------------------------------------------------------------------------------------------------------------------
# 7. Submission: prediction.json = {query_id: [7 candidate ids, best first]} in a zip
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ranked, qids, pool, out_dir, name):
    pred = {}
    for q in qids:
        lst = [c for c in ranked[q] if c != q and c in pool]
        lst = list(dict.fromkeys(lst))[:K_EVAL]
        assert len(lst) == K_EVAL, f"query {q}: fewer than {K_EVAL} candidates"
        pred[q] = lst
    os.makedirs(out_dir, exist_ok=True)
    jp = os.path.join(out_dir, f"{name}_prediction.json"); json.dump(pred, open(jp, "w"))
    errs = validate(json.load(open(jp)), {q: [] for q in qids}, set(pool))           # the course evaluator's checks
    assert not errs, errs[:5]
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(jp, arcname="prediction.json")
    return jp, zp


# ---- the course evaluator (evaluate_phase7.py), functions only ----
#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc


def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)


def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}

## 7. Adaptation techniques

In [7]:
# ===================== Phase 3 adaptation techniques: frozen, full FT, LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix =====================
# All of them leave the Phase-1 model code untouched: they wrap/replace nn.Linear modules inside the backbone, or add small trainable tensors.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------------------------------------------------
# LoRA / rsLoRA / LoRMA
# ----------------------------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2022): y = W0 x + b + s * B A x,  A (r, in) random, B (out, r) zeros -> starts exactly at W0.
    s = alpha / r (LoRA)  or  alpha / sqrt(r) (rank-stabilised LoRA, Kalajdzievski 2023: keeps the update size stable when r grows)."""

    def __init__(self, base, r, alpha, dropout=0.0, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_ROT_IDX = {}


def _row_rotation_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                      # Mpi[i, j] = M[i, (j - i) mod d]  (row i rotated right by i)
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel, Modi, ACL Findings 2025): a MULTIPLICATIVE low-rank update of a frozen W0 (out x in).
      "plus" (LoRMA+):  h = ((alpha/r) B A + I) W0 x + b,   A (r, out) random, B (out, r) zeros -> starts at W0. Computed as
                        y0 = W0 x;  y0 + (alpha/r) B (A y0), so the out x out matrix is never formed.
      "pi"   (LoRMA_pi): h = I_pi(B A) W0 x + b, I_pi rotates row i of BA cyclically by i (rank inflation). Init so that I_pi(BA) = I."""

    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        self.base, self.variant, self.r, self.scale = base, variant, r, alpha / r
        d = base.out_features
        if variant == "plus":
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        elif variant == "pi":
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.randn(d, r) * (1.0 / math.sqrt(d)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
        else:
            raise ValueError(variant)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        else:
            M = self.B @ self.A
            Mpi = torch.gather(M, 1, _row_rotation_index(M.shape[0], M.device))
            y = F.linear(y0, Mpi.to(y0.dtype))
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


# ----------------------------------------------------------------------------------------------------------------------
# AdaLoRA (Zhang et al. 2023), simplified
# ----------------------------------------------------------------------------------------------------------------------
class AdaLoRALinear(nn.Module):
    """y = W0 x + b + s * P diag(lam * mask) Q x   (an SVD-shaped update: P (out, r) and Q (r, in) start orthonormal, lam (r) starts at 0).
    The controller below prunes whole (P_i, lam_i, Q_i) triplets by importance during training, moving the rank budget to the layers that
    need it. Simplification: a pruned triplet stays pruned."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        out, inn = base.out_features, base.in_features
        P = torch.linalg.qr(torch.randn(out, r))[0]                      # orthonormal columns
        Q = torch.linalg.qr(torch.randn(inn, r))[0].t().contiguous()     # orthonormal rows
        self.P, self.Q = nn.Parameter(P), nn.Parameter(Q)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("ada_mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.Q) * (self.lam * self.ada_mask)
        return self.base(x) + self.scale * F.linear(z, self.P)

    def orth_penalty(self):
        eye = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.t() - eye) ** 2).sum()


class AdaLoRAController:
    """Importance score of a triplet i: s(lam_i) + mean_j s(P_ji) + mean_j s(Q_ij), with s(w) = smoothed |w * grad(w)| times its
    smoothed uncertainty (paper, eq. 8-10). Budget schedule: full budget for the first t_i steps, then a cubic decrease to the target
    budget, then fixed for the last t_f steps. Also adds the orthogonality penalty to the loss."""

    def __init__(self, modules, target_rank, ti_frac=0.1, tf_frac=0.2, delta_t=10, beta=0.85, orth_weight=0.1):
        self.mods = modules
        self.r0 = modules[0].r
        self.b0 = len(modules) * self.r0
        self.bT = int(round(len(modules) * target_rank))
        self.ti_frac, self.tf_frac, self.dt, self.beta, self.orth = ti_frac, tf_frac, delta_t, beta, orth_weight
        z = lambda p: torch.zeros_like(p.data)
        self.st = [{n: (z(getattr(m, n)), z(getattr(m, n))) for n in ("lam", "P", "Q")} for m in modules]

    def extra_loss(self):
        return self.orth * sum(m.orth_penalty() for m in self.mods) / len(self.mods)

    def budget(self, step, total):
        ti, tf = int(self.ti_frac * total), int(self.tf_frac * total)
        if step < ti:
            return self.b0
        if step > total - tf:
            return self.bT
        frac = (step - ti) / max(1, total - ti - tf)
        return int(round(self.bT + (self.b0 - self.bT) * (1 - frac) ** 3))

    @torch.no_grad()
    def on_before_step(self, step, total):
        b = self.beta
        for m, st in zip(self.mods, self.st):
            for n, (ibar, u) in list(st.items()):
                p = getattr(m, n)
                if ibar.device != p.device:                       # the statistics were created on the CPU, the model may have moved since
                    ibar, u = ibar.to(p.device), u.to(p.device); st[n] = (ibar, u)
                if p.grad is None:
                    continue
                imp = (p.data * p.grad).abs()
                ibar.mul_(b).add_(imp, alpha=1 - b)
                u.mul_((b)).add_((imp - ibar).abs(), alpha=1 - b)
        tf = int(self.tf_frac * total)
        if step % self.dt == 0 and step <= total - tf:
            scores = []
            for m, st in zip(self.mods, self.st):
                s = lambda n: st[n][0] * st[n][1]
                scores.append(s("lam") + s("P").mean(0) + s("Q").mean(1))
            allsc = torch.cat(scores)
            keep = min(max(self.budget(step, total), 1), allsc.numel())
            thr = torch.sort(allsc, descending=True).values[keep - 1]
            for m, sc in zip(self.mods, scores):
                m.ada_mask.copy_((sc >= thr).float() * m.ada_mask)       # a pruned triplet stays pruned

    def rank_report(self):
        return [int(m.ada_mask.sum().item()) for m in self.mods]


# ----------------------------------------------------------------------------------------------------------------------
# QLoRA (Dettmers et al. 2023): frozen 4-bit NormalFloat base weights (+ double quantisation of the block constants) with LoRA on top
# ----------------------------------------------------------------------------------------------------------------------
NF4_TABLE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
                          -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                          0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """A frozen Linear stored as 4-bit NF4 codes (two per byte) with one absmax constant per `block` weights; the constants are themselves
    stored in 8 bit (double quantisation). The weight is rebuilt on the fly in every forward pass (pure PyTorch: no bitsandbytes)."""

    def __init__(self, base, block=64, double_quant=True, dq_block=256):
        super().__init__()
        self.in_features, self.out_features, self.block, self.double_quant = base.in_features, base.out_features, block, double_quant
        W = base.weight.detach().float().reshape(-1)
        self.numel = W.numel()
        Wp = torch.cat([W, W.new_zeros((-self.numel) % block)])
        blocks = Wp.view(-1, block)
        absmax = blocks.abs().amax(1).clamp(min=1e-8)
        flat = (blocks / absmax[:, None]).reshape(-1)
        codes = torch.empty(flat.numel(), dtype=torch.uint8)
        for i in range(0, flat.numel(), 1 << 21):
            ch = flat[i:i + (1 << 21)]
            codes[i:i + ch.numel()] = (ch.unsqueeze(1) - NF4_TABLE).abs().argmin(1).to(torch.uint8)
        self.register_buffer("packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("nf4", NF4_TABLE.clone(), persistent=False)
        self.n_blocks = absmax.numel()
        if double_quant:
            mean = absmax.mean()
            am = absmax - mean
            amp = torch.cat([am, am.new_zeros((-am.numel()) % dq_block)]).view(-1, dq_block)
            sc = amp.abs().amax(1).clamp(min=1e-12)
            self.register_buffer("am_q", torch.round(amp / sc[:, None] * 127).to(torch.int8))
            self.register_buffer("am_scale", sc)
            self.register_buffer("am_mean", mean.reshape(1))
        else:
            self.register_buffer("absmax", absmax)
        self.register_buffer("bias", base.bias.detach().clone() if base.bias is not None else None)

    def _absmax(self):
        if self.double_quant:
            return (self.am_q.float() / 127 * self.am_scale[:, None]).reshape(-1)[:self.n_blocks] + self.am_mean
        return self.absmax

    @torch.no_grad()
    def dequant(self, dtype=torch.float32):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0x0F).long()], 1).reshape(-1)
        vals = self.nf4[codes].view(-1, self.block) * self._absmax()[:, None]
        return vals.reshape(-1)[:self.numel].view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        W = self.dequant(x.dtype)
        return F.linear(x, W, None if self.bias is None else self.bias.to(x.dtype))


# ----------------------------------------------------------------------------------------------------------------------
# IA3 (Liu et al. 2022): learned per-channel rescaling vectors, initialised to 1
# ----------------------------------------------------------------------------------------------------------------------
class IA3Out(nn.Module):
    """y = l * Linear(x): rescales the OUTPUT channels (used on the key and value projections)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """y = Linear(l * x): rescales the INPUT channels (used on the feed-forward down-projection, i.e. the hidden activation)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


# ----------------------------------------------------------------------------------------------------------------------
# Applying a technique
# ----------------------------------------------------------------------------------------------------------------------
def _swap_linears(root, fn):
    """fn(qualified_name, nn.Linear) -> replacement module or None. Returns the names that were replaced."""
    done = []
    for qual, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear):
                full = f"{qual}.{cname}" if qual else cname
                new = fn(full, child)
                if new is not None:
                    setattr(mod, cname, new); done.append(full)
    return done


def _hit(full, suffixes):
    return any(full.endswith(s) for s in suffixes)


def size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6 + sum(b.numel() * b.element_size() for b in model.buffers()) / 1e6


def apply_technique(model, C, log=print):
    """Freezes/unfreezes the backbone and inserts modules according to C['TECHNIQUE']; the sentence head is always trainable."""
    tech, bb = C["TECHNIQUE"], model.backbone
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    T = C.get("PEFT_TARGETS", [])
    if tech == "frozen":
        pass                                                           # Phase-1 weights untouched: only the head learns (control)
    elif tech == "full_ft":
        for p in bb.parameters():
            p.requires_grad = True
    elif tech in ("lora", "rslora"):
        done = _swap_linears(bb, lambda n, l: LoRALinear(l, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0), tech == "rslora") if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], scale=(C["ALPHA"] / math.sqrt(C["RANK"]) if tech == "rslora" else C["ALPHA"] / C["RANK"]))
    elif tech == "lorma":
        done = _swap_linears(bb, lambda n, l: LoRMALinear(l, C["RANK"], C["ALPHA"], C["LORMA_VARIANT"]) if _hit(n, T) else None)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], variant=C["LORMA_VARIANT"])
    elif tech == "adalora":
        mods = []

        def mk(n, l):
            if _hit(n, T):
                m = AdaLoRALinear(l, C["ADALORA_INIT_RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0)); mods.append(m); return m
        done = _swap_linears(bb, mk)
        assert done, f"no linear layer matches PEFT_TARGETS={T}"
        ctrl = AdaLoRAController(mods, C["ADALORA_TARGET_RANK"], delta_t=C.get("ADALORA_DELTA_T", 10), orth_weight=C.get("ADALORA_ORTH", 0.1))
        model.extra_loss, model.on_before_step, model.ada_controller = ctrl.extra_loss, ctrl.on_before_step, ctrl
        info.update(targets=len(done), init_rank=C["ADALORA_INIT_RANK"], target_rank=C["ADALORA_TARGET_RANK"])
    elif tech == "qlora":
        blocks = bb.layer_list()
        n_q, n_l = [0], [0]

        def mk(n, l):
            q = NF4Linear(l, C.get("NF4_BLOCK", 64), C.get("NF4_DOUBLE_QUANT", True))
            n_q[0] += 1
            if _hit(n, T):
                n_l[0] += 1
                return LoRALinear(q, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0))
            return q
        for blk in blocks:
            _swap_linears(blk, mk)
        assert n_l[0], f"no linear layer matches PEFT_TARGETS={T}"
        info.update(quantized_linears=n_q[0], lora_targets=n_l[0], rank=C["RANK"], alpha=C["ALPHA"])
    elif tech == "ia3":
        n_out, n_in = [0], [0]

        def mk(n, l):
            if _hit(n, C["IA3_OUT"]):
                n_out[0] += 1; return IA3Out(l)
            if _hit(n, C["IA3_IN"]):
                n_in[0] += 1; return IA3In(l)
        _swap_linears(bb, mk)
        assert n_out[0] + n_in[0], f"no linear layer matches IA3_OUT={C['IA3_OUT']} / IA3_IN={C['IA3_IN']}"
        info.update(out_scaled=n_out[0], in_scaled=n_in[0])
    elif tech == "prompt":
        P = C["PROMPT_LEN"]
        emb = bb.embedding_weight()
        idx = torch.randint(100, emb.shape[0], (P,), generator=torch.Generator().manual_seed(0))
        model.soft_prompt = nn.Parameter(emb[idx].detach().clone().float())
        info.update(prompt_len=P)
    elif tech == "prefix":
        bb.enable_prefix(C["PREFIX_LEN"], 0)
        info.update(prefix_len=C["PREFIX_LEN"])
    else:
        raise ValueError(f"unknown TECHNIQUE {tech}")
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); "
        f"stored size {info['model_size_mb']} MB")
    return info

In [8]:
# ===================== Phase 4 additions to the adaptation techniques: BitFit, partial (top-layer) fine-tuning, bottleneck adapters =====================
# Wraps the Phase-3 apply_technique (LoRA, rsLoRA, LoRMA, AdaLoRA, QLoRA, IA3, prompt, prefix, full_ft, frozen are unchanged).
_p3_apply_technique = apply_technique


class BottleneckAdapter(nn.Module):
    """Houlsby/Pfeiffer-style adapter: y + W_up act(W_down y); W_up starts at zero so the model starts exactly at the Phase-1 function."""

    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, y):
        return y + self.up(F.gelu(self.down(y)))


def _attach_adapter(m, d, r):
    ad = BottleneckAdapter(d, r)
    m.add_module("adapter", ad)
    orig = m.forward
    def fwd(*a, **k):
        return m.adapter(orig(*a, **k))
    m.forward = fwd
    return ad


def apply_technique(model, C, log=print):
    tech, bb = C["TECHNIQUE"], model.backbone
    if tech not in ("bitfit", "partial", "adapters"):
        return _p3_apply_technique(model, C, log)
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    if tech == "bitfit":                                   # Ben Zaken et al. 2022: only the bias terms of the backbone are trained
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith("bias") or ".bias_" in name or name.endswith(("bias_ih_l0", "bias_hh_l0")):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "this backbone has no bias terms"
        info.update(bias_params=n)
    elif tech == "partial":                                # layer-wise fine-tuning: the top PARTIAL_LAYERS layers + the final norm
        layers = bb.layer_list(); k = C["PARTIAL_LAYERS"]
        for l in layers[-k:]:
            for p in l.parameters():
                p.requires_grad = True
        for p in bb.final_modules_params():
            p.requires_grad = True
        info.update(unfrozen_layers=k, of_layers=len(layers))
    else:                                                  # adapters
        sites = bb.adapter_sites(C["ADAPTER_KIND"])
        for m, d in sites:
            _attach_adapter(m, d, C["ADAPTER_R"])
        info.update(adapter_kind=C["ADAPTER_KIND"], adapter_sites=len(sites), bottleneck=C["ADAPTER_R"])
    for p in model.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=orig,
                trainable_pct_of_backbone=round(100.0 * trainable / orig, 4), model_size_mb=round(size_mb(model), 1))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {orig:,}); stored size {info['model_size_mb']} MB")
    return info

## 8. Data: Phase_7_PCR, tokens (cached), TF-IDF baseline and hard negatives, header-citation analysis

In [9]:
# checked upload: a failed upload (e.g. a full storage quota) is reported loudly and listed at the end of the notebook
UPLOAD_FAILED = []
_upload_unchecked = upload_model_file
def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    if "batch_bucket_files" in globals() and "HF_BUCKET" in globals():
        try:
            batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN); print(f"uploaded {name} to bucket {HF_BUCKET}"); return True
        except Exception as e:
            print(f"!!! UPLOAD FAILED: {name} ({type(e).__name__}: {str(e)[:200]}). The file is still in {local_path}"); UPLOAD_FAILED.append(local_path); return False
    _upload_unchecked(local_path, name); return True

import numpy as np, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=False)
from sklearn.feature_extraction.text import TfidfVectorizer
P7 = "Phase_7_PCR"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)
Q, CP, REL = {}, {}, {}
for s, S in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
    Q[s] = {str(k): v for k, v in load_json(get_dataset_file(f"{P7}/{S}/{s}.json")).items()}
    CP[s] = {str(k): v for k, v in load_json(get_dataset_file(f"{P7}/{S}/{s}_candidates.json")).items()}
    if s != "test":
        REL[s] = {q: [str(c) for c in v.get("relevant_candidates", []) if isinstance(c, str) and c.strip()] for q, v in Q[s].items()}
if QUICK_TEST:      # a small consistent slice: some queries per split + their relevant candidates + other candidates
    rq = random.Random(0)
    for s in ("train", "dev", "test"):
        qs = sorted(Q[s])[:{"train": 24, "dev": 10, "test": 8}[s]]
        keep = set(qs) | {c for q in qs for c in REL.get(s, {}).get(q, [])} | set(rq.sample(sorted(CP[s]), min(40, len(CP[s]))))
        Q[s] = {q: Q[s][q] for q in qs}; CP[s] = {c: v for c, v in CP[s].items() if c in keep}
        if s in REL: REL[s] = {q: [c for c in REL[s][q] if c in CP[s]] for q in qs}
POOL = {s: sorted(CP[s]) for s in CP}
TEST_QIDS = sorted(Q["test"])
TRAIN_QIDS = [q for q in sorted(Q["train"]) if any(c != q for c in REL["train"][q])]
print({s: (len(Q[s]), len(CP[s])) for s in Q}, f"| {len(TRAIN_QIDS)} training queries with at least one relevant case")
TEXT = {}                                       # id -> raw case (queries and candidates of all splits; the same id has the same text)
for s in Q:
    TEXT.update(CP[s]); TEXT.update(Q[s])
MODES = sorted({o.get("HEADER", BASE["HEADER"]) for _, o in SWEEP} | {BASE["HEADER"]})
assert VOCAB < 65536, "token ids are stored as uint16"


def tokens_for(mode):
    cname = f"p7tok_{mode}_{MAX_DOC_TOKENS}.pt"; ids = sorted(TEXT)
    key = hashlib.md5("|".join(ids).encode()).hexdigest()
    if not QUICK_TEST:
        try:
            c = torch.load(fetch_model_file(cname), map_location="cpu", weights_only=False)
            if c["key"] == key: print(f"  tokens ({mode}) loaded from the bucket cache"); return c["toks"]
        except Exception:
            pass
    t0 = time.time(); out = {}
    for k, i in enumerate(ids):
        out[i] = encode_case(bpe_tokenizer, case_text(TEXT[i], mode), MAX_DOC_TOKENS)
        if (k + 1) % 2000 == 0: print(f"    {mode}: {k + 1}/{len(ids)} cases tokenised [{time.time() - t0:.0f}s]")
    if not QUICK_TEST:
        torch.save(dict(key=key, toks=out), f"{WORK}/{cname}"); upload_model_file(f"{WORK}/{cname}", cname)
    return out
TOKS = {m: tokens_for(m) for m in MODES}
n = np.array([len(v) for v in TOKS[MODES[0]].values()])
print(f"cases tokenised: {len(n)}; tokens median {int(np.median(n))}, p90 {int(np.percentile(n, 90))}; cut at {MAX_DOC_TOKENS}: {(n >= MAX_DOC_TOKENS).mean():.1%}")

# ---- TF-IDF (lexical) scores: hard negatives for training, a baseline, and the hybrid ----
def tfidf_matrix(s, mode):
    qids, cids = (TRAIN_QIDS if s == "train" else sorted(Q[s])), POOL[s]
    tv = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), sublinear_tf=True, min_df=2 if not QUICK_TEST else 1, max_df=0.9, dtype=np.float32)
    X = tv.fit_transform([case_text(TEXT[c], mode) for c in cids]); Qm = tv.transform([case_text(TEXT[q], mode) for q in qids])
    return qids, cids, (Qm @ X.T).toarray().astype(np.float32)
TFIDF, HARD = {}, {}
t0 = time.time()
for mode in MODES:
    for s in ("train", "dev", "test"):
        TFIDF[(s, mode)] = tfidf_matrix(s, mode)
    qids, cids, S = TFIDF[("train", mode)]
    HARD[mode] = rank_from_scores(S, qids, cids, k=200)
    for s in ("dev",):
        qids, cids, S = TFIDF[(s, mode)]
        m = retrieval_scores(REL[s], rank_from_scores(S, qids, cids))
        print(f"TF-IDF baseline ({mode} header), {s}: F1@7 {m['f1']:.4f} P@7 {m['p']:.4f} R@7 {m['r']:.4f} R@100 {m['r@100']:.3f} MRR {m['mrr']:.4f}")
    m = retrieval_scores({q: REL["train"][q] for q in TRAIN_QIDS}, HARD[mode])
    print(f"TF-IDF baseline ({mode} header), train: F1@7 {m['f1']:.4f}; hard negatives = its top {BASE['HARD_TOP']} non-relevant cases [{time.time() - t0:.0f}s]")

# ---- how much do the header citations explain? (link = the query's own AIR citation is in the candidate's CITATOR INFO, or vice versa) ----
CIT = {i: citation_keys(doc_text(TEXT[i])) for i in TEXT}
def linked(q, c):
    (oq, rq), (oc, rc) = CIT[q], CIT[c]
    return bool(oq & rc) or bool(oc & rq)
for s in ("train", "dev"):
    qs = [q for q in REL[s] if REL[s][q]]
    pairs = [(q, c) for q in qs for c in REL[s][q]]
    n_link = sum(linked(q, c) for q, c in pairs)
    allp = sum(1 for q in qs for c in POOL[s] if c != q and linked(q, c))
    has = np.mean([bool(CIT[q][0]) for q in qs])
    print(f"citation links ({s}): {has:.0%} of queries have an AIR citation in their header; {n_link}/{len(pairs)} relevant pairs ({n_link / max(1, len(pairs)):.1%}) are linked by the header; "
          f"precision of the link rule {n_link / max(1, allp):.3f} ({allp} linked pairs in the pool)")
    if s == "dev":
        qids, cids, S = TFIDF[("dev", "keep")] if ("dev", "keep") in TFIDF else TFIDF[("dev", MODES[0])]
        S2 = S.copy(); pos = {c: j for j, c in enumerate(cids)}
        for i, q in enumerate(qids):
            for c in cids:
                if linked(q, c): S2[i, pos[c]] += 10.0
        m = retrieval_scores(REL[s], rank_from_scores(S2, qids, cids))
        print(f"  rule 'linked cases first, then TF-IDF' on dev: F1@7 {m['f1']:.4f} P@7 {m['p']:.4f} R@7 {m['r']:.4f} (a header artefact, reported for the analysis, never submitted)")
SAN_DOCS = {i: {"ids": TOKS[MODES[0]][i][:1025].astype(np.int64).tolist()} for i in TRAIN_QIDS[:8]}

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

... (2 outputs left out)

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

{'train': (827, 4320), 'dev': (118, 1023), 'test': (237, 1727)} | 817 training queries with at least one relevant case


  tokens (keep) loaded from the bucket cache
cases tokenised: 5825; tokens median 4104, p90 13786; cut at 16384: 7.7%
TF-IDF baseline (keep header), dev: F1@7 0.3969 P@7 0.4322 R@7 0.3669 R@100 0.710 MRR 0.8681
TF-IDF baseline (keep header), train: F1@7 0.3152; hard negatives = its top 50 non-relevant cases [89s]
citation links (train): 29% of queries have an AIR citation in their header; 503/5308 relevant pairs (9.5%) are linked by the header; precision of the link rule 0.592 (850 linked pairs in the pool)
citation links (dev): 29% of queries have an AIR citation in their header; 105/973 relevant pairs (10.8%) are linked by the header; precision of the link rule 0.814 (129 linked pairs in the pool)
  rule 'linked cases first, then TF-IDF' on dev: F1@7 0.4136 P@7 0.4504 R@7 0.3823 (a header artefact, reported for the analysis, never submitted)


## 9. Load the Phase-1 checkpoint, define how a model is built

In [10]:
t0 = time.time()
_ck = torch.load(fetch_model_file(PHASE1_CHECKPOINT), map_location="cpu", mmap=True)
PHASE1_STATE = _ck["model"] if "model" in _ck else _ck
del _ck
_lm = build_phase1_lm()
PHASE1_PARAMS = sum(p.numel() for p in _lm.parameters())
SANITY_LOSS = lm_sanity_loss(_lm, SAN_DOCS, device)
print(f"Phase-1 model {MODEL_KEY}: {PHASE1_PARAMS:,} parameters; next-token loss on the case texts = {SANITY_LOSS:.3f} nats "
      f"(far below {math.log(VOCAB):.1f} = trained weights loaded; about {math.log(VOCAB):.1f} = random weights) [{time.time()-t0:.0f}s]")
assert SANITY_LOSS < 8.0, "the Phase-1 weights do not seem to be loaded correctly"
del _lm; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None


def new_backbone(lm):
    return DecoderBackbone(lm, bidirectional=False)


def build_model(C, log=print):
    """Phase-1 model -> backbone -> case encoder (chunk pooling + projection) -> adaptation technique."""
    if C.get("RANDOM_INIT"):     # control: same architecture with freshly initialised weights (the Phase-1 weights are NOT loaded)
        _orig_lsd = nn.Module.load_state_dict
        nn.Module.load_state_dict = lambda self, *a, **k: None
        try:
            lm = build_phase1_lm()
        finally:
            nn.Module.load_state_dict = _orig_lsd
    else:
        lm = build_phase1_lm()
    bb = new_backbone(lm)
    model = CaseEncoder(bb, bb.d_model, C)
    info = apply_technique(model, C, log)
    return model.to(device), info


def free():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

phase1_decoder_only_gqa_kvcache_full_lay(…): reconstructing file:   0%|          |  0.00B / 1.33GB            

phase1_decoder_only_gqa_kvcache_full_lay(…): downloading bytes:           |  0.00B            

(found phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt in the old repo kavincg23/cs781-checkpoints)
Phase-1 model gqa160: 111,095,872 parameters; next-token loss on the case texts = 3.667 nats (far below 10.6 = trained weights loaded; about 10.6 = random weights) [12s]


## 10. Train every setup

In [11]:
RES_DIR = f"{WORK}/p7res_{SWEEP_NAME}"; os.makedirs(RES_DIR, exist_ok=True)
SUB_DIR = f"{WORK}/p7_submissions_{SWEEP_NAME}"; os.makedirs(SUB_DIR, exist_ok=True)
SWEEP_START = time.time()


def make_C(ov):
    C = dict(BASE, **ov)
    if "LR" not in ov: C["LR"] = LR_BY_TECH.get(C["TECHNIQUE"], C["LR"])
    if C["TECHNIQUE"] == "prompt": C["WINDOW"] = min(C["WINDOW"], 2048 - C["PROMPT_LEN"])
    return C


def cfg_key(C):
    return hashlib.md5(json.dumps([C, len(TRAIN_QIDS), QUICK_TEST, MAX_DOC_TOKENS], sort_keys=True, default=str).encode()).hexdigest()[:10]


def cache_name(n): return f"p7res_{SWEEP_NAME}__{n}.pt"
def cache_load(n, key):
    try:
        r = torch.load(fetch_model_file(cache_name(n), f"{RES_DIR}/{cache_name(n)}"), map_location="cpu", weights_only=False)
        return r if r.get("key") == key else None
    except Exception:
        return None


def wait_for(fname):
    t_w = time.time()
    while True:
        try:
            return fetch_model_file(fname)
        except Exception:
            if (time.time() - t_w) / 60 >= TEACHER_WAIT_MIN: raise FileNotFoundError(f"teacher file {fname} is not in the bucket (run the teacher notebook first)")
            print(f"  waiting for the teacher file {fname} ({(time.time() - t_w) / 60:.0f} min so far)"); time.sleep(300)


def dense_rank(E, split, C):
    qids = sorted(Q[split]) if split != "train" else TRAIN_QIDS
    S = score_matrix(E, qids, POOL[split], C["SIM"], device)
    return qids, S


def evaluator(C):
    toks = TOKS[C["HEADER"]]; ids = sorted(set(Q["dev"]) | set(POOL["dev"]))
    def ev(model):
        E = encode_cases(model, toks, ids, C, device, C["INFER_CHUNKS"])
        qids, S = dense_rank(E, "dev", C)
        return retrieval_scores(REL["dev"], rank_from_scores(S, qids, POOL["dev"]))
    return ev


def run_setup(name, ov):
    C = make_C(ov); key = cfg_key(C)
    r = cache_load(name, key)
    if r is not None:
        print(f"[{name}] cached: dev F1@7 {r['dev']['f1']:.4f} (hybrid {r['dev_hybrid']['f1']:.4f})"); return r
    print(f"== {name}: {ov}")
    teacher = None
    if C["KD_TEACHER"]:
        tf = torch.load(wait_for(C["KD_TEACHER"]), map_location="cpu", weights_only=False)
        teacher = {i: v.float() for i, v in tf["emb"].items()}
        print(f"  KD teacher: {C['KD_TEACHER']} ({tf.get('setup')}, {tf.get('params', 0) / 1e6:.1f}M parameters, dim {next(iter(teacher.values())).shape[0]}), mode {C['KD_MODE']}, alpha {C['KD_ALPHA']}")
        C["TEACHER_TAU"] = tf.get("tau", C["TEACHER_TAU"])
    model, info = build_model(C)
    data = dict(qids=TRAIN_QIDS, rel=REL["train"], pool=POOL["train"], toks=TOKS[C["HEADER"]], hard=HARD[C["HEADER"]])
    if teacher is not None:
        data["qids"] = [q for q in TRAIN_QIDS if q in teacher]
    res = train_retriever(model, data, C, device, evaluator(C), teacher=teacher)
    if res["best_state"] is not None: load_trainable_state(model, res["best_state"])
    # ---- final: index dev and test (INFER_CHUNKS per case), dense ranking, hybrid with TF-IDF (lambda chosen on dev) ----
    toks = TOKS[C["HEADER"]]; t0 = time.time()
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
    ids_dev = sorted(set(Q["dev"]) | set(POOL["dev"])); ids_test = sorted(set(Q["test"]) | set(POOL["test"]))
    E_dev = encode_cases(model, toks, ids_dev, C, device, C["INFER_CHUNKS"]); t_dev = time.time() - t0
    E_test = encode_cases(model, toks, ids_test, C, device, C["INFER_CHUNKS"])
    infer_peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    qd, Sd = dense_rank(E_dev, "dev", C); qt, St = dense_rank(E_test, "test", C)
    dev = retrieval_scores(REL["dev"], rank_from_scores(Sd, qd, POOL["dev"]))
    _, _, Td = TFIDF[("dev", C["HEADER"])]; _, _, Tt = TFIDF[("test", C["HEADER"])]
    lam_res = {lam: retrieval_scores(REL["dev"], rank_from_scores(hybrid_scores(Sd, Td, lam), qd, POOL["dev"]))["f1"] for lam in HYBRID_LAMBDAS}
    lam = max(lam_res, key=lam_res.get)
    dev_h = retrieval_scores(REL["dev"], rank_from_scores(hybrid_scores(Sd, Td, lam), qd, POOL["dev"]))
    test_rank = rank_from_scores(St, qt, POOL["test"]); test_rank_h = rank_from_scores(hybrid_scores(St, Tt, lam), qt, POOL["test"])
    try:      # the course evaluator's own scoring on dev
        jp = f"{RES_DIR}/{name}_dev_prediction.json"; rk = rank_from_scores(Sd, qd, POOL["dev"])
        json.dump({q: [c for c in rk[q] if c != q][:7] for q in qd}, open(jp, "w"))
        off = score(load_truth(Path(get_dataset_file(f"{P7}/Dev/dev.json"))) if not QUICK_TEST else REL["dev"], json.load(open(jp)))
        print(f"  [official evaluator] dev Micro-F1@7 {off['primary_metric_1']:.4f} P@7 {off['secondary_metric_1']:.4f} R@7 {off['secondary_metric_2']:.4f}")
    except Exception as e:
        print(f"  (official evaluator check skipped: {type(e).__name__}: {str(e)[:120]})")
    if C["SAVE_TRAIN_EMB"]:     # teacher file for KD: case vectors of every training query and training candidate
        ids_tr = sorted(set(TRAIN_QIDS) | set(POOL["train"])); t1 = time.time()
        E_tr = encode_cases(model, toks, ids_tr, C, device, C["TRAIN_CHUNKS"])
        if not isinstance(next(iter(E_tr.values())), tuple):
            tname = f"p7emb_{SWEEP_NAME}__{name}.pt"
            torch.save(dict(emb=E_tr, setup=name, params=info["params_total"], tau=C["TAU"], dim=model.out_dim, header=C["HEADER"]), f"{RES_DIR}/{tname}")
            upload_model_file(f"{RES_DIR}/{tname}", tname); print(f"  teacher vectors for {len(E_tr)} training cases written ({(time.time() - t1) / 60:.1f} min)")
    if C["SAVE_WEIGHTS"]:
        wname = f"p7w_{SWEEP_NAME}__{name}.pt"
        torch.save(dict(setup=name, overrides=ov, C=C, state=res["best_state"], best_epoch=res["best_epoch"]), f"{RES_DIR}/{wname}"); upload_model_file(f"{RES_DIR}/{wname}", wname)
    dim = model.out_dim; per_case = (dim * 2 * (C["INFER_CHUNKS"] if C["DOC_POOL"] == "maxsim" else 1))
    r = dict(key=key, name=name, overrides=ov, C=C, info=info, history=res["history"], best_epoch=res["best_epoch"], epoch0=res["epoch0"], dev=dev, dev_hybrid=dev_h, hybrid_lambda=lam,
             hybrid_by_lambda=lam_res, train_seconds=res["train_seconds"], peak_mem_gb=res["peak_mem_gb"], infer_peak_gb=infer_peak, train_docs_per_sec=res["docs_per_sec"],
             index_docs_per_sec=round(len(ids_dev) / max(t_dev, 1e-9), 2), index_bytes_per_case=per_case, dim=dim,
             dev_scores_dense=Sd.astype(np.float16), dev_qids=qd, test_scores_dense=St.astype(np.float16), test_qids=qt,
             test_rank={q: test_rank[q][:50] for q in qt}, test_rank_hybrid={q: test_rank_h[q][:50] for q in qt},
             emb_dev=E_dev if not isinstance(next(iter(E_dev.values())), tuple) else None, emb_test=E_test if not isinstance(next(iter(E_test.values())), tuple) else None)
    torch.save(r, f"{RES_DIR}/{cache_name(name)}"); upload_model_file(f"{RES_DIR}/{cache_name(name)}", cache_name(name))
    print(f"### {name}: dev F1@7 {dev['f1']:.4f} P@7 {dev['p']:.4f} R@7 {dev['r']:.4f} | R@100 {dev['r@100']:.3f} MRR {dev['mrr']:.4f} | hybrid with TF-IDF (lambda {lam}) F1@7 {dev_h['f1']:.4f} | "
          f"best epoch {res['best_epoch']} | train {res['train_seconds'] / 60:.1f} min, peak {res['peak_mem_gb']} GB | indexing {r['index_docs_per_sec']:.1f} cases/s")
    del model; free()
    return r


RESULTS, SETUP_MIN, FAILED = {}, [], []
for name, ov in SWEEP:
    elapsed = (time.time() - SWEEP_START) / 60; need = 1.15 * max(SETUP_MIN) if SETUP_MIN else 0
    if elapsed + need > MAX_SWEEP_HOURS * 60 and cache_load(name, cfg_key(make_C(ov))) is None:
        print(f"{elapsed:.0f} min used, a setup needs about {need:.0f} min: not starting {name} (Run All again in a new session to continue)"); continue
    t_s = time.time()
    try:
        RESULTS[name] = run_setup(name, ov)
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append(name); print(f"!!! setup {name} FAILED ({type(e).__name__}: {str(e)[:200]}); continuing with the next one"); free()
    if time.time() - t_s > 60: SETUP_MIN.append((time.time() - t_s) / 60)
print(f"sweep done in {(time.time() - SWEEP_START) / 60:.0f} min" + (f"; FAILED: {FAILED}" if FAILED else ""))

== gqa_uni_list: {'TECHNIQUE': 'lorma', 'SELECT': 'uniform', 'OBJECTIVE': 'listwise'}
Technique lorma: trainable 688,384 of 81,024,256 parameters (0.6196% of the Phase-1 model's 111,095,872); stored size 325.1 MB
training: 817 queries per epoch, 8 per step (103 steps), 6 epochs; objective listwise, 1 hard negative(s) per query + in-batch; 4 chunks per case in training, 8 at inference; 688,384 trainable parameters
  epoch 0 (before training, Phase-1 weights as they are): dev F1@7 0.2624 R@100 0.668 MRR 0.6470
  epoch 1: train_loss=2.2212, dev_f1=0.2924, dev_p=0.3184, dev_r=0.2703, dev_r100=0.69, dev_mrr=0.6281, train_min=5.2, eval_min=0.8
  epoch 2: train_loss=2.0037, dev_f1=0.3124, dev_p=0.3402, dev_r=0.2888, dev_r100=0.682, dev_mrr=0.6756, train_min=5.4, eval_min=0.8
  epoch 3: train_loss=1.8658, dev_f1=0.3191, dev_p=0.3475, dev_r=0.295, dev_r100=0.7, dev_mrr=0.6571, train_min=5.3, eval_min=0.8
  epoch 4: train_loss=1.7108, dev_f1=0.3079, dev_p=0.3354, dev_r=0.2847, dev_r100=0.69, dev

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p7res_p7_gqa2__gqa_uni_list.pt to bucket kavincg23/cs781-models
### gqa_uni_list: dev F1@7 0.3191 P@7 0.3475 R@7 0.2950 | R@100 0.700 MRR 0.6571 | hybrid with TF-IDF (lambda 2.0) F1@7 0.4024 | best epoch 3 | train 32.0 min, peak 1.256 GB | indexing 22.0 cases/s
== gqa_uni_list_proj0: {'TECHNIQUE': 'lorma', 'SELECT': 'uniform', 'OBJECTIVE': 'listwise', 'PROJ_DIM': 0}
Technique lorma: trainable 491,520 of 80,827,392 parameters (0.4424% of the Phase-1 model's 111,095,872); stored size 324.4 MB
training: 817 queries per epoch, 8 per step (103 steps), 6 epochs; objective listwise, 1 hard negative(s) per query + in-batch; 4 chunks per case in training, 8 at inference; 491,520 trainable parameters
  epoch 0 (before training, Phase-1 weights as they are): dev F1@7 0.2702 R@100 0.671 MRR 0.6409
  epoch 1: train_loss=2.2553, dev_f1=0.3091, dev_p=0.3366, dev_r=0.2857, dev_r100=0.683, dev_mrr=0.6964, train_min=5.4, eval_min=0.8
  epoch 2: train_loss=2.072, dev_f1=0.3257, dev_p=0.3547, dev

Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (5 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p7res_p7_gqa2__gqa_uni_list_8ch.pt to bucket kavincg23/cs781-models
### gqa_uni_list_8ch: dev F1@7 0.2991 P@7 0.3257 R@7 0.2765 | R@100 0.670 MRR 0.6111 | hybrid with TF-IDF (lambda 2.0) F1@7 0.4069 | best epoch 4 | train 57.3 min, peak 1.255 GB | indexing 22.1 cases/s
sweep done in 144 min


## 11. Comparison table

In [12]:
rows = sorted(RESULTS.items(), key=lambda kv: -kv[1]["dev"]["f1"])
print(f"Dev = {len(Q['dev'])} dev queries ranked against the dev pool ({len(POOL['dev'])} cases), the query itself excluded. F1/P/R = Micro @7 (primary: F1); "
      f"ep0 = F1@7 before training (Phase-1 weights as they are); hybrid = dense + lambda x TF-IDF, lambda chosen on dev.")
print(f"{'setup':26s} {'F1@7':>6s} {'P@7':>6s} {'R@7':>6s} {'R@100':>6s} {'MRR':>6s} {'ep0':>6s} {'hybrid':>7s} {'lam':>4s} {'ep':>3s} {'train M':>8s} {'%bb':>6s} {'GB':>5s} {'min':>6s} {'idx c/s':>7s} {'B/case':>7s}")
for n, r in rows:
    d, h, i = r["dev"], r["dev_hybrid"], r["info"]; e0 = r["epoch0"]["f1"] if r.get("epoch0") else float("nan")
    print(f"{n:26s} {d['f1']:6.4f} {d['p']:6.4f} {d['r']:6.4f} {d['r@100']:6.3f} {d['mrr']:6.4f} {e0:6.4f} {h['f1']:7.4f} {r['hybrid_lambda']:4.2g} {r['best_epoch']:3d} "
          f"{i['params_trainable'] / 1e6:8.2f} {i['trainable_pct_of_backbone']:6.2f} {(r['peak_mem_gb'] or 0):5.1f} {r['train_seconds'] / 60:6.1f} {r['index_docs_per_sec']:7.1f} {r['index_bytes_per_case']:7d}")
for mode in MODES:
    qids, cids, S = TFIDF[("dev", mode)]; m = retrieval_scores(REL["dev"], rank_from_scores(S, qids, cids))
    print(f"{'TF-IDF baseline (' + mode + ')':26s} {m['f1']:6.4f} {m['p']:6.4f} {m['r']:6.4f} {m['r@100']:6.3f} {m['mrr']:6.4f}")
print("train M = trainable parameters (millions); %bb = share of the Phase-1 model; GB = peak training memory; min = training minutes; idx c/s = cases indexed per second (dev); B/case = index bytes per case (fp16).")
for n, r in rows:
    print(f"  {n}: per-epoch dev F1@7 {[h['dev_f1'] for h in r['history']]}; hybrid F1@7 by lambda {r['hybrid_by_lambda']}")

Dev = 118 dev queries ranked against the dev pool (1023 cases), the query itself excluded. F1/P/R = Micro @7 (primary: F1); ep0 = F1@7 before training (Phase-1 weights as they are); hybrid = dense + lambda x TF-IDF, lambda chosen on dev.
setup                        F1@7    P@7    R@7  R@100    MRR    ep0  hybrid  lam  ep  train M    %bb    GB    min idx c/s  B/case
gqa_uni_list_proj0         0.3346 0.3644 0.3094  0.704 0.7028 0.2702  0.4013    4   5     0.49   0.44   1.3   32.1    21.9    1536
gqa_uni_list               0.3191 0.3475 0.2950  0.700 0.6571 0.2624  0.4024    2   3     0.69   0.62   1.3   32.0    22.0     512
gqa_uni_list_8ch           0.2991 0.3257 0.2765  0.670 0.6111 0.2624  0.4069    2   4     0.69   0.62   1.3   57.3    22.1     512
TF-IDF baseline (keep)     0.3969 0.4322 0.3669  0.710 0.8681
train M = trainable parameters (millions); %bb = share of the Phase-1 model; GB = peak training memory; min = training minutes; idx c/s = cases indexed per second (dev); B/case

## 12. Submission files

In [13]:
if SUBMIT or QUICK_TEST:
    for n, r in rows:
        for kind, rk, f1 in (("dense", r["test_rank"], r["dev"]["f1"]), ("hybrid", r["test_rank_hybrid"], r["dev_hybrid"]["f1"])):
            jp, zp = write_submission(rk, TEST_QIDS, set(POOL["test"]), SUB_DIR, f"{SWEEP_NAME}_{n}_{kind}")
            print(f"{os.path.basename(zp):62s} dev F1@7 {f1:.4f}; file check (course evaluator rules) passed"); upload_model_file(zp)
else:
    print("SUBMIT = False: no submission files from this notebook (comparison runs).")

submission_p7_gqa2_gqa_uni_list_proj0_dense.zip                dev F1@7 0.3346; file check (course evaluator rules) passed


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded submission_p7_gqa2_gqa_uni_list_proj0_dense.zip to bucket kavincg23/cs781-models
submission_p7_gqa2_gqa_uni_list_proj0_hybrid.zip               dev F1@7 0.4013; file check (course evaluator rules) passed


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (15 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded submission_p7_gqa2_gqa_uni_list_8ch_hybrid.zip to bucket kavincg23/cs781-models


## 13. Summary

In [14]:
SUMMARY = dict(sweep=SWEEP_NAME, model=MODEL_KEY, phase1_checkpoint=PHASE1_CHECKPOINT, phase1_params=PHASE1_PARAMS, sanity_loss=SANITY_LOSS, base=BASE,
               results={n: {k: v for k, v in r.items() if k not in ("dev_scores_dense", "test_scores_dense", "test_rank", "test_rank_hybrid", "dev_qids", "test_qids", "emb_dev", "emb_test")} for n, r in RESULTS.items()},
               total_minutes=round((time.time() - NB_START) / 60, 1))
path = f"{WORK}/summary_{SWEEP_NAME}.json"
json.dump(SUMMARY, open(path, "w"), indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
upload_model_file(path)
for p in sorted(os.listdir(SUB_DIR)):
    print("  ", f"{SUB_DIR}/{p}", os.path.getsize(f"{SUB_DIR}/{p}") // 1024, "KB")
if UPLOAD_FAILED:
    print("!" * 80 + f"\n{len(UPLOAD_FAILED)} FILES COULD NOT BE UPLOADED: " + ", ".join(UPLOAD_FAILED) + "\n" + "!" * 80)
print(f"done in {(time.time() - NB_START) / 60:.0f} min. Send back the whole output (the tables above).")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded summary_p7_gqa2.json to bucket kavincg23/cs781-models
   /kaggle/working/p7_submissions_p7_gqa2/p7_gqa2_gqa_uni_list_8ch_dense_prediction.json 26 KB
   /kaggle/working/p7_submissions_p7_gqa2/p7_gqa2_gqa_uni_list_8ch_hybrid_prediction.json 26 KB
   /kaggle/working/p7_submissions_p7_gqa2/p7_gqa2_gqa_uni_list_dense_prediction.json 26 KB
   /kaggle/working/p7_submissions_p7_gqa2/p7_gqa2_gqa_uni_list_hybrid_prediction.json 26 KB
   /kaggle/working/p7_submissions_p7_gqa2/p7_gqa2_gqa_uni_list_proj0_dense_prediction.json 26 KB
   /kaggle/working/p7_submissions_p7_gqa2/p7_gqa2_gqa_uni_list_proj0_hybrid_prediction.json 26 KB
   /kaggle/working/p7_submissions_p7_gqa2/submission_p7_gqa2_gqa_uni_list_8ch_dense.zip 7 KB
   /kaggle/working/p7_submissions_p7_gqa2/submission_p7_gqa2_gqa_uni_list_8ch_hybrid.zip 7 KB
   /kaggle/working/p7_submissions_p7_gqa2/submission_p7_gqa2_gqa_uni_list_dense.zip 7 KB
   /kaggle/working/p7_submissions_p7_gqa2/submission_p7_gqa2_gqa_uni_list_hybrid.zip 7 KB
  